# 实验 1：成对排序 (Pairwise Ranking)

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('.'))
import time
import numpy as np
import matplotlib.pyplot as plt
from models.ranking import generate_ranking_data
from algorithms.admm import run_u_admm
from utils.excel_utils import append_to_excel
from utils.eval_utils import evaluate_ranking_accuracy, calculate_metrics

# 1. 统一集中定义参数
params = {
    'Experiment': 'Pairwise Ranking',
    'm': 10, 
    'n': 100, # ⚠️ 注意：U-统计量是 O(n^2) 复杂度。n=200 的计算量是 n=100 的 4 倍！调参时建议用 100
    'p_prime': 5, 
    'p': 20, 
    'pc': 0.3,
    'T': 40, 
    'W_inner': 10, 
    'rho': 2.3, 
    'lam_t': 0.04,
    'noise_type': 'normal',
    'rng_seed': 479,
    'run_baselines': False  # 设为 False 可以跳过耗时的 Pooled MR 和 D-subGD
}
np.random.seed(params['rng_seed'])

# 2. 生成数据
d_rank = generate_ranking_data(
    m=params['m'], n=params['n'], p_prime=params['p_prime'], 
    p=params['p'], pc=params['pc'], noise_type=params['noise_type'], rng_seed=params['rng_seed']
)
theta_true = d_rank['theta_true']

# 3. 运行 Proposed (U-ADMM)
t0 = time.time()
theta_u_r, theta_n_r, hist_r = run_u_admm(
    d_rank, T=params['T'], W_inner=params['W_inner'], 
    rho=params['rho'], lam_t=params['lam_t'], verbose=True
)
time_uadmm = time.time() - t0
theta_uadmm = theta_u_r[0]
print(f'Proposed 耗时: {time_uadmm:.1f}s')

# 4. 运行其他基线算法
theta_avg = theta_n_r
rmse_local, rmse_global, rmse_dgd = 0.0, 0.0, 0.0
time_global, time_dgd = 0.0, 0.0

if params['run_baselines']:
    from algorithms.admm import init_all_nodes
    theta0_list = init_all_nodes(d_rank)
    local_rmses = [calculate_metrics(theta_true, th)[0] for th in theta0_list]
    rmse_local = np.mean(local_rmses)
    
    from algorithms.baselines import run_global_u_erm, run_dgd
    t0 = time.time()
    theta_global = run_global_u_erm(d_rank)
    time_global = time.time() - t0
    rmse_global, _ = calculate_metrics(theta_true, theta_global)
    print(f'Pooled MR 耗时: {time_global:.1f}s')
    
    t0 = time.time()
    theta_dgd = run_dgd(d_rank, T=params['T'] * params['W_inner'], lr=0.1)
    time_dgd = time.time() - t0
    rmse_dgd, _ = calculate_metrics(theta_true, theta_dgd)
    print(f'D-subGD 耗时: {time_dgd:.1f}s')


In [ ]:
# 绘制收敛曲线
plt.figure(figsize=(8, 5))
plt.plot(hist_r['rmse'], marker='o', label='U-ADMM RMSE')
rmse_naive = np.linalg.norm(theta_n_r - d_rank['theta_true'])
plt.axhline(rmse_naive, color='r', linestyle='--', label='Naive RMSE')
plt.xlabel('Outer Iteration t')
plt.ylabel('RMSE')
plt.title('Convergence of U-ADMM (Ranking)')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# 绘制系数比较图
plt.figure(figsize=(8, 5))
plt.plot(d_rank['theta_true'], marker='o', linestyle='None', label='True Coefficients')
plt.plot(theta_u_r[0], marker='x', color='r', linestyle='None', label='Estimated Coefficients')
plt.xlabel('Coefficient Index')
plt.ylabel('Coefficient Value')
plt.title('Coefficient Comparison (Ranking)')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# 计算指标并保存到 Excel
rmse_uadmm, mae_uadmm = calculate_metrics(theta_true, theta_uadmm)
rmse_avg, _ = calculate_metrics(theta_true, theta_avg)
class_metrics = evaluate_ranking_accuracy(d_rank['X'], d_rank['Y'], theta_uadmm, d_rank['quantiles'])

print(f"\n=== RMSE 结果 ({params['noise_type']} 噪声) ===")
if params['run_baselines']:
    print(f"1. Pooled MR (Global): {rmse_global:.4f}")
    print(f"2. Local MR:         {rmse_local:.4f}")
print(f"3. Avg MR (Naive):   {rmse_avg:.4f}")
if params['run_baselines']:
    print(f"4. D-subGD:          {rmse_dgd:.4f}")
print(f"5. Proposed (U-ADMM): {rmse_uadmm:.4f}")
print(f"\n成对准确率 (Proposed): {class_metrics['Pairwise_Accuracy']:.2%}")

record = params.copy()
record['RMSE'] = rmse_uadmm
record['MAE'] = mae_uadmm
record['Avg_RMSE'] = rmse_avg
if params['run_baselines']:
    record['Pooled_RMSE'] = rmse_global
    record['Local_RMSE'] = rmse_local
    record['DGD_RMSE'] = rmse_dgd

# 记录第一个节点的所有系数估计
for i, coef in enumerate(theta_uadmm.flatten()):
    record[f'theta_{i}'] = coef

record.update(class_metrics)

os.makedirs('exp1', exist_ok=True)
append_to_excel('exp1/experiment_results.xlsx', record)
print('结果已保存到 exp1/experiment_results.xlsx')
